# Music Hit Prediction

We aim to predict the probability that a track is a music hit using track metadata and audio features. The competition evaluates predictions using ROC AUC.

This notebook contains the preparation steps we have retained so far. We encode the 1,000 most frequent artist genres and represent missing genre information explicitly. We also fill missing track titles, album names, and record labels with `unknown`. We remove surrounding spaces from artist names and replace release dates with release years. We preserve all other values and the original row order.

All preparation code is included here. The only inputs are the original `train.csv`, `test.csv`, and `sample_submission.csv`; the notebook does not depend on our exploratory notebooks, reports, processed CSV, or saved JSON vocabulary.

## 1. Configuration and imports

We use pandas and NumPy for data handling and SciPy for sparse genre indicators. `DATA_DIR` identifies the folder containing the three original CSV files. We support the repository layout as well as CSV files placed beside this notebook.

In [1]:
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from IPython.display import display

TOP_N_GENRES = 1000
CSV_FILES = ("train.csv", "test.csv", "sample_submission.csv")
DATA_DIR = next(
    (directory for directory in (
        Path.cwd() / "data/raw",
        Path.cwd().parent / "data/raw",
        Path.cwd(),
    ) if all((directory / name).is_file() for name in CSV_FILES)),
    None,
)
if DATA_DIR is None:
    raise FileNotFoundError("Set DATA_DIR to the folder containing the three original CSV files.")

# Optional local export of the complete preparation; not required for model training.
EXPORT_PROCESSED_CSV = True
PROCESSED_DIR = DATA_DIR.parent / "processed" if DATA_DIR.name == "raw" else DATA_DIR / "processed"


## 2. Load the original data

We use the labeled training data to select genres. The test data contains no target, and the sample submission provides the expected identifiers and output format.

In [2]:
train_raw = pd.read_csv(DATA_DIR / "train.csv")
test_raw = pd.read_csv(DATA_DIR / "test.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

assert "music_hit" in train_raw.columns
assert "music_hit" not in test_raw.columns
assert train_raw.columns.drop("music_hit").equals(test_raw.columns)
assert sample_submission.columns.tolist() == ["id", "music_hit"]
assert sample_submission["id"].equals(test_raw["id"])

display(pd.DataFrame({
    "dataset": ["train", "test", "sample_submission"],
    "rows": [len(train_raw), len(test_raw), len(sample_submission)],
    "columns": [train_raw.shape[1], test_raw.shape[1], sample_submission.shape[1]],
}))


,dataset,rows,columns
0,train,350051,24
1,test,87513,23
2,sample_submission,87513,2


## 3. Study and fill missing text metadata

We examined missing values in `track_name`, `album_name`, and `record_label`. We fill null values, empty strings, and whitespace-only fields with `unknown` in both datasets. These few missing fields do not justify discarding tracks that still contain other useful features.

We preserve existing non-missing names and every row. This rule does not use the target or statistics learned from either dataset; the choice to include these text columns in a model remains separate.

In [3]:
TEXT_COLUMNS = ("track_name", "album_name", "record_label")
MISSING_TEXT_VALUE = "unknown"


def missing_text_mask(series):
    return series.isna() | series.astype("string").str.strip().eq("").fillna(False)


def summarize_text_metadata(datasets):
    rows = []
    for name, frame in datasets.items():
        for column in TEXT_COLUMNS:
            missing = missing_text_mask(frame[column])
            rows.append({
                "dataset": name,
                "column": column,
                "missing values": int(missing.sum()),
                "% missing": round(100 * missing.mean(), 4),
                "distinct non-missing values": frame.loc[~missing, column].nunique(),
            })
    return pd.DataFrame(rows)


def fill_missing_text(frame):
    cleaned = frame.copy()
    for column in TEXT_COLUMNS:
        cleaned.loc[missing_text_mask(cleaned[column]), column] = MISSING_TEXT_VALUE
    return cleaned

missing_text_summary = summarize_text_metadata({"train": train_raw, "test": test_raw})
display(missing_text_summary)


,dataset,column,missing values,% missing,distinct non-missing values
0,train,track_name,4,0.0011,248383
1,train,album_name,1,0.0003,57310
2,train,record_label,38,0.0109,31558
3,test,track_name,1,0.0011,73246
4,test,album_name,0,0.0000,32169
5,test,record_label,14,0.0160,17780


In [4]:
train_clean = fill_missing_text(train_raw)
test_clean = fill_missing_text(test_raw)

for original, cleaned in ((train_raw, train_clean), (test_raw, test_clean)):
    pd.testing.assert_frame_equal(
        original.drop(columns=list(TEXT_COLUMNS)),
        cleaned.drop(columns=list(TEXT_COLUMNS)),
    )
    for column in TEXT_COLUMNS:
        missing = missing_text_mask(original[column])
        assert not missing_text_mask(cleaned[column]).any()
        assert cleaned.loc[missing, column].eq(MISSING_TEXT_VALUE).all()
        pd.testing.assert_series_equal(original.loc[~missing, column], cleaned.loc[~missing, column])
print("Filled missing text values; all rows and non-missing values were preserved.")


Filled missing text values; all rows and non-missing values were preserved.


## 4. Study and trim artist names

We found surrounding whitespace in 34 training rows and 9 test rows. The affected names are `Workout Music ` and `Raven Black `; their follower counts are consistent within each name, and neither has an existing version without the trailing space.

We remove leading and trailing spaces in both datasets. We preserve internal spaces, capitalization, accents, missing values, and follower counts. This is a text standardization step; we do not infer artist identity or fill genres from names.

In [5]:
def artist_name_whitespace_mask(values):
    return values.notna() & values.ne(values.str.strip()).fillna(False)


def summarize_artist_whitespace(datasets):
    return pd.DataFrame([
        {
            "dataset": name,
            "rows with surrounding spaces": int(artist_name_whitespace_mask(frame["artist_name"]).sum()),
            "distinct affected names": frame.loc[
                artist_name_whitespace_mask(frame["artist_name"]), "artist_name"
            ].nunique(),
        }
        for name, frame in datasets.items()
    ])


def trim_artist_names(frame):
    cleaned = frame.copy()
    cleaned["artist_name"] = cleaned["artist_name"].str.strip()
    return cleaned

artist_whitespace_summary = summarize_artist_whitespace({"train": train_raw, "test": test_raw})
display(artist_whitespace_summary)

artist_names = pd.concat([
    train_raw[["artist_name", "artist_followers"]],
    test_raw[["artist_name", "artist_followers"]],
], ignore_index=True)
artist_names["trimmed_name"] = artist_names["artist_name"].str.strip()
changed_names = artist_names.loc[
    artist_name_whitespace_mask(artist_names["artist_name"]), "trimmed_name"
].drop_duplicates()
artist_name_examples = []
for name in sorted(changed_names):
    group = artist_names[artist_names["trimmed_name"].eq(name)]
    artist_name_examples.append({
        "original names (repr)": ", ".join(repr(value) for value in sorted(group["artist_name"].unique())),
        "trimmed name": name,
        "tracks across train and test": len(group),
        "distinct follower counts": group["artist_followers"].nunique(),
        "follower counts": ", ".join(str(value) for value in sorted(group["artist_followers"].unique())),
        "rows already without spaces": int(group["artist_name"].eq(name).sum()),
    })
display(pd.DataFrame(artist_name_examples))


,dataset,rows with surrounding spaces,distinct affected names
0,train,34,2
1,test,9,2


,original names (repr),trimmed name,tracks across train and test,distinct follower counts,follower counts,rows already without spaces
0,'Raven Black ',Raven Black,11,1,21127,0
1,'Workout Music ',Workout Music,32,1,72649,0


In [6]:
train_clean = trim_artist_names(train_clean)
test_clean = trim_artist_names(test_clean)
for original, cleaned in ((train_raw, train_clean), (test_raw, test_clean)):
    pd.testing.assert_series_equal(original["artist_name"].str.strip(), cleaned["artist_name"])
    pd.testing.assert_series_equal(original["artist_followers"], cleaned["artist_followers"])
    assert not artist_name_whitespace_mask(cleaned["artist_name"]).any()
print("Removed surrounding spaces from artist names; follower counts were preserved.")


Removed surrounding spaces from artist names; follower counts were preserved.


## 5. Study release dates and retain the year

We examined validity, date ranges, and the concentration on January 1. Approximately 11.7% of training dates fall on January 1, suggesting possible default month/day values without establishing that the dates are approximate.

We initially retain only `release_year` to represent release periods, replacing `release_date` in both datasets. We use a nullable integer so missing or invalid dates would remain missing. The usefulness of additional date components remains to be assessed in validation.

In [7]:
def parse_release_dates(values):
    return pd.to_datetime(values, format="%Y-%m-%d", errors="coerce")


def summarize_release_dates(datasets):
    rows = []
    for name, frame in datasets.items():
        values = frame["release_date"]
        dates = parse_release_dates(values)
        missing = values.isna() | values.astype("string").str.strip().eq("").fillna(False)
        valid_count = int(dates.notna().sum())
        january_first = int(((dates.dt.month == 1) & (dates.dt.day == 1)).sum())
        rows.append({
            "dataset": name,
            "missing dates": int(missing.sum()),
            "invalid non-missing dates": int((~missing & dates.isna()).sum()),
            "earliest date": dates.min().date().isoformat() if valid_count else None,
            "latest date": dates.max().date().isoformat() if valid_count else None,
            "distinct years": dates.dt.year.nunique(),
            "January 1 tracks": january_first,
            "% January 1 among valid dates": round(100 * january_first / valid_count, 2) if valid_count else 0.0,
        })
    return pd.DataFrame(rows)


def extract_release_year(frame):
    cleaned = frame.copy()
    position = cleaned.columns.get_loc("release_date")
    years = parse_release_dates(cleaned["release_date"]).dt.year.astype("Int16")
    cleaned = cleaned.drop(columns="release_date")
    cleaned.insert(position, "release_year", years)
    return cleaned

release_date_summary = summarize_release_dates({"train": train_raw, "test": test_raw})
display(release_date_summary)


,dataset,missing dates,invalid non-missing dates,earliest date,latest date,distinct years,January 1 tracks,% January 1 among valid dates
0,train,0,0,1956-01-01,2023-12-22,68,40863,11.67
1,test,0,0,1956-01-01,2023-12-22,68,10147,11.59


In [8]:
train_clean = extract_release_year(train_clean)
test_clean = extract_release_year(test_clean)
for original, cleaned in ((train_raw, train_clean), (test_raw, test_clean)):
    assert "release_date" not in cleaned.columns
    expected_years = parse_release_dates(original["release_date"]).dt.year.astype("Int16").rename("release_year")
    pd.testing.assert_series_equal(cleaned["release_year"], expected_years)
print("Replaced release_date with the numeric release_year feature in training and test data.")


Replaced release_date with the numeric release_year feature in training and test data.


## 6. Study and select artist genres

We examined genre frequencies and compared vocabulary sizes before retaining the 1,000 most frequent genres. We used only the training data for this analysis. Frequencies count tracks rather than unique artists; a genre is counted at most once per track.

### 6.1. Missing information and most frequent genres

We split genre lists on `|` and count the individual genres. Tracks may have several genres, so their percentages do not sum to 100%.

In [9]:
def parse_genres(value):
    if pd.isna(value):
        return set()
    return {genre.strip() for genre in value.split("|") if genre.strip()}


def select_genres(values, top_n):
    if top_n < 1:
        raise ValueError("top_n must be positive.")
    counts = Counter()
    for value in values:
        counts.update(parse_genres(value))
    if len(counts) < top_n:
        raise ValueError(f"Requested {top_n} genres, but found only {len(counts)}.")
    selected = sorted(counts, key=lambda genre: (-counts[genre], genre))[:top_n]
    return selected, counts

genre_counts = Counter()
genre_combinations = Counter()
for value in train_raw["artist_genres"]:
    genres = frozenset(parse_genres(value))
    genre_counts.update(genres)
    genre_combinations[genres] += 1

missing_genre_tracks = genre_combinations.get(frozenset(), 0)
known_tracks = len(train_raw) - missing_genre_tracks
ranked_genres = sorted(genre_counts, key=lambda genre: (-genre_counts[genre], genre))
print(f"Distinct genres: {len(genre_counts):,}")
print(f"Tracks without genres: {missing_genre_tracks:,} ({100 * missing_genre_tracks / len(train_raw):.2f}%)")

frequent_genres = pd.DataFrame([
    {"genre": genre, "tracks": genre_counts[genre],
     "% of training tracks": round(100 * genre_counts[genre] / len(train_raw), 2)}
    for genre in ranked_genres[:10]
])
display(frequent_genres)


Distinct genres: 3,886
Tracks without genres: 162,592 (46.45%)


,genre,tracks,% of training tracks
0,karaoke,13491,3.85
1,rap,12524,3.58
2,pop,12240,3.50
3,rock,11691,3.34
4,hip hop,9231,2.64
5,classical,7327,2.09
6,pop rap,6908,1.97
7,dance pop,6489,1.85
8,trap,4942,1.41
9,classic rock,4661,1.33


### 6.2. Coverage for different vocabulary sizes

We measured coverage among tracks with known genres. **At least one genre retained** means that a track has at least one selected genre; **all genres retained** means that its complete genre list is represented. We excluded tracks without genres from both percentages.

In [10]:
coverage_rows = []
for size in sorted({100, 500, TOP_N_GENRES, len(ranked_genres)}):
    selected_set = set(ranked_genres[:size])
    covered = sum(count for genres, count in genre_combinations.items()
                  if genres and genres & selected_set)
    fully_covered = sum(count for genres, count in genre_combinations.items()
                        if genres and genres <= selected_set)
    coverage_rows.append({
        "genre columns": min(size, len(ranked_genres)),
        "% with at least one genre retained": round(100 * covered / known_tracks, 2),
        "% with all genres retained": round(100 * fully_covered / known_tracks, 2),
    })
display(pd.DataFrame(coverage_rows))


,genre columns,% with at least one genre retained,% with all genres retained
0,100,52.46,32.99
1,500,77.66,66.54
2,1000,88.70,81.37
3,3886,100.00,100.00


### 6.3. Initial decision

We retained the 1,000 most frequent genres, reducing the vocabulary from 3,886 columns while preserving at least one genre for 88.70% of tracks with known genres and all genres for 81.37%. In comparison, 500 genres retain at least one genre for 77.66% of these tracks. We represent missing genre information with a separate indicator.

This is our initial trade-off between coverage and the number of features. Its impact on model performance remains to be evaluated using validation ROC AUC. When introducing a validation split, we will fit vocabulary selection on that split's training portion only.

In [11]:
selected_genres, _ = select_genres(train_raw["artist_genres"], TOP_N_GENRES)
print(f"Selected genre columns: {len(selected_genres):,}, plus genre_unknown.")


Selected genre columns: 1,000, plus genre_unknown.


## 7. Encode genres consistently

We replace `artist_genres` with one binary column per selected genre and `genre_unknown`. A selected genre is 1 when present and 0 otherwise. `genre_unknown` is 1 only when the original genre information is missing or empty.

Known genres outside our selected vocabulary, including unseen test genres, contribute no selected indicators and do not activate `genre_unknown`. We use the same columns and order for both datasets. Sparse storage avoids allocating a dense table containing mostly zeros.

In [12]:
def encode_genres(frame, vocabulary):
    genre_columns = [f"genre_{genre}" for genre in vocabulary] + ["genre_unknown"]
    other_columns = frame.columns.drop("artist_genres").tolist()
    if len(set(genre_columns + other_columns)) != len(genre_columns + other_columns):
        raise ValueError("Genre column names must be unique and distinct from existing columns.")
    positions = {genre: index for index, genre in enumerate(vocabulary)}
    indices = []
    indptr = [0]
    for value in frame["artist_genres"]:
        genres = parse_genres(value)
        if not genres:
            indices.append(len(vocabulary))
        else:
            indices.extend(sorted(positions[genre] for genre in genres if genre in positions))
        indptr.append(len(indices))

    matrix = csr_matrix(
        (np.ones(len(indices), dtype=np.uint8),
         np.asarray(indices, dtype=np.int32),
         np.asarray(indptr, dtype=np.int32)),
        shape=(len(frame), len(genre_columns)),
    )
    encoded = pd.DataFrame.sparse.from_spmatrix(
        matrix, index=frame.index, columns=genre_columns,
    )
    genre_index = frame.columns.get_loc("artist_genres")
    return pd.concat([
        frame.iloc[:, :genre_index],
        encoded,
        frame.iloc[:, genre_index + 1:],
    ], axis=1)


train_prepared = encode_genres(train_clean, selected_genres)
test_prepared = encode_genres(test_clean, selected_genres)


## 8. Check the prepared data

We verify that the prepared training and test data have matching feature columns, that missing genres are flagged correctly, and that all other values are preserved apart from missing-text filling, artist-name trimming, and the replacement of release dates with years. The genre indicators use sparse storage in memory. We can export the complete preparation to CSV in the following optional step.

In [13]:
genre_columns = [f"genre_{genre}" for genre in selected_genres] + ["genre_unknown"]
assert train_prepared.columns.drop("music_hit").equals(test_prepared.columns)
for original, prepared in ((train_raw, train_prepared), (test_raw, test_prepared)):
    expected_metadata = extract_release_year(
        trim_artist_names(fill_missing_text(original))
    ).drop(columns="artist_genres")
    pd.testing.assert_frame_equal(prepared[expected_metadata.columns], expected_metadata)
    assert "release_date" not in prepared.columns
    assert prepared["release_year"].dtype == pd.Int16Dtype()
    expected_unknown = original["artist_genres"].map(lambda value: int(not parse_genres(value)))
    np.testing.assert_array_equal(
        prepared["genre_unknown"].sparse.to_dense().to_numpy(),
        expected_unknown.to_numpy(),
    )
    assert all(prepared[column].dtype == pd.SparseDtype("uint8", 0) for column in genre_columns)

display(pd.DataFrame({
    "dataset": ["train_prepared", "test_prepared"],
    "rows": [len(train_prepared), len(test_prepared)],
    "columns": [train_prepared.shape[1], test_prepared.shape[1]],
    "genre_unknown": [int(train_prepared["genre_unknown"].sum()),
                      int(test_prepared["genre_unknown"].sum())],
}))


,dataset,rows,columns,genre_unknown
0,train_prepared,350051,1024,162592
1,test_prepared,87513,1023,40733


## 9. Export the complete prepared datasets

We export `train_prepared` and `test_prepared` to `data/processed/train.csv` and `data/processed/test.csv` in the repository layout. Both include missing-text filling, artist-name trimming, release-year extraction, and the same training-derived genre vocabulary.

This optional local export is controlled by `EXPORT_PROCESSED_CSV`. We write in chunks and replace each destination only after its temporary CSV is complete. The raw CSV files remain unchanged. These large intermediate files are excluded from Git; they are not the final prediction submission.

In [14]:
if EXPORT_PROCESSED_CSV:
    PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
    exports = []
    for name, frame in (("train", train_prepared), ("test", test_prepared)):
        destination = PROCESSED_DIR / f"{name}.csv"
        if destination.resolve() == (DATA_DIR / f"{name}.csv").resolve():
            raise ValueError("Processed exports must not overwrite the raw CSV files.")
        temporary = destination.with_suffix(".tmp.csv")
        frame.to_csv(temporary, index=False, chunksize=5000)
        temporary.replace(destination)
        saved_ids = pd.read_csv(destination, usecols=["id"])["id"]
        pd.testing.assert_series_equal(saved_ids, frame["id"].reset_index(drop=True))
        exports.append({"file": destination.name, "rows": len(frame), "columns": frame.shape[1]})
        print(f"Saved {destination.name}: {len(frame):,} rows, {frame.shape[1]:,} columns")
    display(pd.DataFrame(exports))
else:
    print("Processed CSV export disabled; prepared frames remain available in memory.")


Saved train.csv: 350,051 rows, 1,024 columns
Saved test.csv: 87,513 rows, 1,023 columns


,file,rows,columns
0,train.csv,350051,1024
1,test.csv,87513,1023


## Current status

We have retained genre encoding, missing-text filling, artist-name whitespace trimming, and release-year extraction. We have not yet added validation, model training, or prediction generation. Our next stage is to evaluate this preparation with a model. The final prediction CSV will contain `id` and the predicted `music_hit` probability for every test track.